In [94]:
import os
from PIL import Image
from utils.paths import path_data_cholec_original_masks, path_results
import numpy as np
import torch
import torch.nn.functional as F
from evaluation.dice import compute_dice
import einops
from tqdm   import tqdm

In [95]:
def load_prediction(prediction_path):
    prediction = Image.open(prediction_path).resize((256,256))
    assert np.max(prediction) <= 13, "Label has values outside expected range [0,13]"
    prediction = prediction.resize((256,256), resample=Image.NEAREST)
    prediction = torch.from_numpy(np.array(prediction)).long()
    prediction = F.one_hot(prediction, num_classes=13).permute(2,0,1).float()
    prediction = prediction[1:]
    prediction = einops.rearrange(prediction, 'c h w -> 1 c h w')
    return prediction

In [96]:
predictions_path = "/local/scratch/sharvien/SASVi/CholecSeg8K_SASVI_Mask2Former_V3_WITHOUT_GT"

In [97]:
dices = []
videos = os.listdir(predictions_path)
for video in tqdm(videos):
    start_frames = sorted(os.listdir(os.path.join(path_data_cholec_original_masks, video)))

    for start_frame in start_frames:

        frames = sorted(os.listdir(os.path.join(path_data_cholec_original_masks, video, start_frame)))
        for frame in frames:
            frame_idx = int(frame.split("_")[1])
            prediction_path = os.path.join(predictions_path, video, f"{frame_idx:010d}_rgb_mask.png")
            gt_path = os.path.join(path_data_cholec_original_masks, video, start_frame, frame)
            if not os.path.exists(prediction_path):
                continue
            pred = load_prediction(prediction_path)
            gt = load_prediction(gt_path)
            pred = 2000 * pred - 1000 # convert to logits
            dices.append(compute_dice(pred, gt))

100%|██████████| 17/17 [00:15<00:00,  1.09it/s]


In [98]:
dices = torch.cat(dices, dim=0)

In [99]:

print((dices.nanmean(dim=0) * 100).int())

tensor([86, 78, 61, 83, 47, 91, 72, 87, 78, 66, 52, 98], dtype=torch.int32)


In [100]:
save_path = os.path.join(os.path.join(path_results, "CholecDataset", "SASVI"), f"dices_val.pth")
os.makedirs(os.path.dirname(save_path), exist_ok=True)
torch.save(dices, save_path)